# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import pickle
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI

for _env in (Path('.env'), Path('mp1-prompt-lab/.env')):
    if _env.exists():
        load_dotenv(_env)
        break

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in mp1-prompt-lab/.env'

client = AsyncOpenAI(base_url=os.environ.get('OPENAI_BASE_URL') or None)

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('data')   # mp1-prompt-lab/data (snippets + golden set)

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
JSON_FIELDS = 'company, role, years_experience_required'


def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [{
        'role': 'user',
        'content': (
            f'Extract {JSON_FIELDS} from this job snippet as JSON. '
            'years_experience_required must be an integer or null if not stated.\n\n'
            f'{snippet_text}'
        ),
    }]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples (not from the eval set)."""
    examples = (
        'Snippet: Maple Street Bakery needs a Junior Baker. One year of shop experience required.\n'
        'JSON: {"company": "Maple Street Bakery", "role": "Junior Baker", "years_experience_required": 1}\n\n'
        'Snippet: Orion Labs is hiring a Staff Hardware Engineer. We hire on demonstrated impact and do not list a years requirement.\n'
        'JSON: {"company": "Orion Labs", "role": "Staff Hardware Engineer", "years_experience_required": null}\n\n'
        'Snippet: Redwood Civic Trust seeks a Grant Writer. Fresh graduates welcome; no prior experience required.\n'
        'JSON: {"company": "Redwood Civic Trust", "role": "Grant Writer", "years_experience_required": 0}\n\n'
    )
    return [{
        'role': 'user',
        'content': (
            f'Extract {JSON_FIELDS} as JSON. Use null when years are not stated. '
            'Use 0 when the posting says no experience is required. Follow the examples.\n\n'
            f'{examples}'
            f'Snippet: {snippet_text}\nJSON:'
        ),
    }]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Persona + explicit JSON schema."""
    return [
        {
            'role': 'system',
            'content': (
                'You are an expert recruiter extracting structured fields from job postings. '
                'Return only a JSON object with exactly these keys: '
                'company (string), role (string), years_experience_required (integer or null). '
                'Rules: take the stated minimum (around 6 → 6, 7+ → 7, 3-5 → 3). '
                'Fresh grads / no experience required → 0. '
                'If years are not stated, return null — do not infer or guess. '
                'Do not wrap the JSON in markdown.'
            ),
        },
        {'role': 'user', 'content': snippet_text},
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [{
        'role': 'user',
        'content': (
            'Think step by step about company name, job title, and minimum years of experience. '
            'Then output a JSON object with keys company, role, years_experience_required '
            '(integer or null if not stated).\n\n'
            f'{snippet_text}'
        ),
    }]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
import re


def parse_response(text: str) -> dict | None:
    """Extract a JSON object from the model response. Return None if it doesn't parse.

    Handles markdown fences and CoT prose before/after the object.
    """
    if text is None:
        return None
    raw = str(text).strip()
    if not raw:
        return None

    fenced = re.search(r'```(?:json)?\s*(.*?)\s*```', raw, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        raw = fenced.group(1).strip()

    start = raw.find('{')
    end = raw.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        obj = json.loads(raw[start:end + 1])
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def _usage_cost(model: str, usage) -> float:
    if usage is None:
        return 0.0
    rates = RATES[model]
    return usage.prompt_tokens * rates['in'] + usage.completion_tokens * rates['out']


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    messages = STRATEGIES[strategy_name](snippet['snippet'])
    t0 = time.perf_counter()
    resp = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    latency_s = time.perf_counter() - t0
    raw = resp.choices[0].message.content or ''
    extracted = parse_response(raw)
    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'raw_response': raw,
        'extracted': extracted,
        'parse_success': extracted is not None,
        'cost_usd': _usage_cost(MODEL, resp.usage),
        'latency_s': latency_s,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(name, snippet)
        for name in STRATEGIES
        for snippet in snippets
    ]
    return list(await asyncio.gather(*tasks))

In [5]:
CACHE_PATH = Path('results_cache.pkl')

if CACHE_PATH.exists():
    results = pickle.loads(CACHE_PATH.read_bytes())
    print(f'Loaded {len(results)} cached extraction results from {CACHE_PATH}.')
else:
    results = await run_all()
    CACHE_PATH.write_bytes(pickle.dumps(results))
    print(f'Got {len(results)} results and wrote {CACHE_PATH}.')

print(f'Total extraction cost: ${sum(r["cost_usd"] for r in results):.4f}')
results[0]

Got 40 results and wrote results_cache.pkl.
Total extraction cost: $0.0020


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'parse_success': True,
 'cost_usd': 3.1349999999999996e-05,
 'latency_s': 2.654726666980423}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [6]:
import re


def _norm_str(value) -> str | None:
    if value is None:
        return None
    text = ' '.join(str(value).split()).casefold()
    if text in {'', 'null', 'none'}:
        return None
    return text


def _norm_years(value):
    """Map 5, '5+', '5 years' → 5. Map missing/null → None."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return int(value)
    text = str(value).strip().casefold()
    if text in {'', 'null', 'none', 'n/a', 'na'}:
        return None
    match = re.search(r'-?\d+', text)
    if not match:
        return text
    return int(match.group(0))


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if not isinstance(extracted, dict):
        return 0
    score = 0
    if _norm_str(extracted.get('company')) == _norm_str(gold.get('company')):
        score += 1
    if _norm_str(extracted.get('role')) == _norm_str(gold.get('role')):
        score += 1
    if _norm_years(extracted.get('years_experience_required')) == _norm_years(gold.get('years_experience_required')):
        score += 1
    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4."""
    payload = {
        'snippet': snippet_text,
        'extracted': extracted,
        'gold': {
            'company': gold.get('company'),
            'role': gold.get('role'),
            'years_experience_required': gold.get('years_experience_required'),
        },
    }
    resp = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        messages=[
            {
                'role': 'system',
                'content': (
                    'You score job-field extraction. Reply with a single integer 1-4 and nothing else.\n'
                    '4 = all three fields correct\n'
                    '3 = two of three correct, and no fabricated years/company/role\n'
                    '2 = one of three correct, or a field was fabricated\n'
                    '1 = none correct or unparsable'
                ),
            },
            {'role': 'user', 'content': json.dumps(payload, default=str)},
        ],
    )
    text = resp.choices[0].message.content or ''
    match = re.search(r'[1-4]', text)
    return int(match.group(0)) if match else 1

In [7]:
# Offline checks for 7a / 7b — no API calls
assert parse_response('{"company": "Acme Corp", "role": "x", "years_experience_required": 5}')['company'] == 'Acme Corp'
assert parse_response('```json\n{"a": 1}\n```') == {'a': 1}
assert parse_response('Reasoning here... {"a": 1}') == {'a': 1}
assert parse_response('not json') is None

gold_j01 = {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5}
assert score_accuracy(
    {'company': 'acme corp', 'role': ' Senior Software Engineer ', 'years_experience_required': '5+'},
    gold_j01,
) == 3

gold_j10 = {'company': 'Cyberdyne Systems', 'role': 'AI/ML Research Scientist', 'years_experience_required': None}
assert score_accuracy({**gold_j10, 'years_experience_required': 5}, gold_j10) == 2
assert score_accuracy(gold_j10, gold_j10) == 3
assert score_accuracy(None, gold_j01) == 0
print('7a parse_response and 7b score_accuracy checks passed.')

7a parse_response and 7b score_accuracy checks passed.


In [8]:
SCORED_CACHE = Path('scored_cache.pkl')
snippet_by_id = {row['id']: row['snippet'] for row in snippets}


async def score_row(row: dict) -> dict:
    gold = golden[row['snippet_id']]
    out = dict(row)
    out['parse_success'] = row.get('extracted') is not None
    out['accuracy'] = score_accuracy(row.get('extracted'), gold)
    out['llm_judge_score'] = await score_llm_judge(
        snippet_by_id[row['snippet_id']],
        row.get('extracted'),
        gold,
    )
    return out


if SCORED_CACHE.exists():
    scored = pickle.loads(SCORED_CACHE.read_bytes())
    print(f'Loaded {len(scored)} cached scored rows from {SCORED_CACHE}.')
else:
    scored = list(await asyncio.gather(*[score_row(row) for row in results]))
    SCORED_CACHE.write_bytes(pickle.dumps(scored))
    print(f'Scored {len(scored)} results and wrote {SCORED_CACHE}.')

Scored 40 results and wrote scored_cache.pkl.


## Step 5 — Build the comparison table

In [9]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']


def _field_hit(row, field) -> bool:
    extracted = row.get('extracted') if isinstance(row.get('extracted'), dict) else None
    gold = golden[row['snippet_id']]
    if extracted is None:
        return False
    if field == 'years_experience_required':
        return _norm_years(extracted.get(field)) == _norm_years(gold.get(field))
    return _norm_str(extracted.get(field)) == _norm_str(gold.get(field))


by_field = pd.DataFrame({
    'company': [_field_hit(r, 'company') for r in scored],
    'role': [_field_hit(r, 'role') for r in scored],
    'years': [_field_hit(r, 'years_experience_required') for r in scored],
    'strategy': [r['strategy'] for r in scored],
}).groupby('strategy').mean().round(3)

print(summary.to_markdown())
print()
print('By-field hit rate')
print(by_field.to_markdown())
summary


| strategy   |   Accuracy (mean of 3) |   Parse rate |   Judge score |   Total cost ($) |   Latency p50 (s) |
|:-----------|-----------------------:|-------------:|--------------:|-----------------:|------------------:|
| cot        |                    2.9 |            1 |           4   |            0.001 |             3.285 |
| few_shot   |                    2.9 |            1 |           3.9 |            0     |             2.247 |
| structured |                    2.8 |            1 |           3.9 |            0     |             2.18  |
| zero_shot  |                    2.7 |            1 |           4   |            0     |             2.197 |

By-field hit rate
| strategy   |   company |   role |   years |
|:-----------|----------:|-------:|--------:|
| cot        |       0.9 |      1 |     1   |
| few_shot   |       1   |      1 |     0.9 |
| structured |       0.9 |      1 |     0.9 |
| zero_shot  |       0.8 |      1 |     0.9 |


,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.9,1.0,4.0,0.001,3.285
few_shot,2.9,1.0,3.9,0.000,2.247
structured,2.8,1.0,3.9,0.000,2.180
zero_shot,2.7,1.0,4.0,0.000,2.197


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```